# Script to fetch Tourplay rosters from tournament urls

In [ ]:
import numpy as np
import pandas as pd
import json
import time

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
#options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

In [ ]:
from PIL import Image
from IPython.display import display

## Fetch rosters for a list of tourneys

In [ ]:
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


In [ ]:
url_list = [ 'https://tourplay.net/en/blood-bowl/welsh-national-championship-2026/'
    
]


In [ ]:
# Note maximize Chrome browser before running this part

ids = []
coaches = []
urls = []

for url in url_list:
    print(url)
    driver.get(url + 'classifications')
    time.sleep(3)
    elements = driver.find_elements(By.CLASS_NAME, 'user-name-to-show')
    cnt = len(elements)
    print(str(cnt) + " rosters available at url")
    
    for counter in range(0,33):
        ids.append(counter)
        elements = driver.find_elements(By.CLASS_NAME, 'user-name-to-show')
        el = elements[counter]
        coaches.append(el.text)
        el.click()
        time.sleep(2)
        print(driver.current_url)
        urls.append(driver.current_url)
        driver.back()
        time.sleep(3)

# quit Chrome Browser session
driver.quit()


In [ ]:
len(df_rosters_to_fetch)

In [ ]:
# create list of tuples
data = zip(ids, coaches, urls)
# create dataframe from list
df_rosters_to_fetch = pd.DataFrame(data, columns=['row_id', 'coach_name', 'roster_url'])
df_rosters_to_fetch['roster_id'] = df_rosters_to_fetch['roster_url'].transform(lambda x: x.split("/")[-1])
df_rosters_to_fetch.to_csv('rosters_to_fetch_A.csv')

## Fetch rosters

In [ ]:
df_rosters_to_fetch = pd.concat([pd.read_csv('rosters_to_fetch_A.csv'), 
                                pd.read_csv('rosters_to_fetch_B.csv')#,
                                #pd.read_csv('rosters_to_fetch_C.csv'),
                                #pd.read_csv('rosters_to_fetch_D.csv')
                                ], ignore_index=True)

In [ ]:
df_rosters_to_fetch.shape

In [ ]:
%run src/get_team_roster.py

# first roster id to fetch
roster_id = df_rosters_to_fetch.iloc[0]['roster_id']

df_roster, roster = get_tourplay_roster(roster_id)

In [ ]:
df_roster


In [ ]:
%run src/get_team_roster.py

cnt = 0
for roster_id in df_rosters_to_fetch['roster_id']:
    if cnt == 0:
        df_roster, roster = get_tourplay_roster(roster_id)
        df_rosters = df_roster
    else:
        df_roster, roster = get_tourplay_roster(roster_id)
        df_rosters = pd.concat([df_rosters, df_roster], ignore_index=True)
    cnt = cnt + 1
    print(".", end = '')

In [ ]:
import os
target = 'datasets/current/df_rosters_welsh_nationals_2026'
os.makedirs(target, exist_ok=True)

df_rosters.to_csv(target + '.csv')

In [ ]:
df_rosters.shape